# 向量化五会话标签：恢复/半日市/两臂身份矩阵

固定复用四个既有Qlib机械时钟。新向量化核心对SOXX、等beta SOXL、半日市双向配置及合成零整股订单，与已修正逐事件引擎按同状态、同Tiered和同分钟高水位逐项对账。2021/2022价值不参与，零新模型/资金候选账户。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_event_checkpoint import capture_after_open
from src.backtest.stage1_capfree_lock_episode import replay_capfree_lock_episode
from src.analysis.stage1_capfree_lock_vectorized import compile_lock_price_plane,value_lock_event
from src.backtest.ibkr_tiered import settle_tiered_event
from src.workflow.stage1_dated_controller import DatedModel,DatedStage1Controllers
from src.workflow.stage1_f696_training_reference import prepare as prepare_f696_source
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from copy import deepcopy
root=Path.cwd();card=root/"research/stage1_v5_lock_vectorized_clock_matrix_20261030";plan=json.loads((card/"plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
parent=root/plan["parent_card"];old=root/plan["old_source_card"]
assert sha(parent/"summary.json")==plan["parent_summary_sha256"] and sha(parent/"verification_manifest.json")==plan["parent_manifest_sha256"]
assert sha(old/"plan.json")==plan["old_source_plan_sha256"]
assert sha(root/plan["checkpoint_code"])==plan["checkpoint_code_sha256"]
assert sha(root/plan["reference_module"])==plan["reference_module_sha256"]
assert sha(root/plan["single_arm_parity_card"]/"audit.json")==plan["single_arm_audit_sha256"]
assert sha(root/plan["minute_provider_uri"]/"qlib_store_manifest.json")==plan["minute_provider_manifest_sha256"]
parent_plan=json.loads((parent/"plan.json").read_text());old_plan=json.loads((old/"plan.json").read_text())
prepared=prepare_f696_source(root=root,card=old,plan=old_plan)
qlib.init(provider_uri=str(root/plan["minute_provider_uri"]),region="us",kernels=1)
names=["analysis/native/dated_f696_0","analysis/controller_audit",*[f"models/open_{y}" for y in parent_plan["signal_years"]],*[f"models/recovery_{y}" for y in (2020,2021,2022)]]
objects,receipt=load_verified_recorder_objects(parent,expected_recorder_id=plan["parent_recorder_id"],artifacts=names,expected_manifest_sha256=plan["parent_manifest_sha256"])
book=objects["analysis/native/dated_f696_0"];frame=prepared["frame"]
assert len(frame)==1056 and book["audit"]["status"]=="passed"
open_models={y:DatedModel(objects[f"models/open_{y}"],f"{y-1}-12-31",f"{parent_plan['action_recorder_id']}:open_action_value_models/{y}") for y in parent_plan["signal_years"]}
recovery_models={}
for y in parent_plan["signal_years"]:
 if y in parent_plan["explicit_no_recovery_model_years"]:recovery_models[y]=DatedModel(None,f"{y-1}-12-31",f"explicit_no_model_{y}_insufficient_prior_teacher",explicit_no_model=True)
 else:recovery_models[y]=DatedModel(objects[f"models/recovery_{y}"],f"{y-1}-12-31",f"{plan['parent_recorder_id']}:models/recovery_{y}")
controller=DatedStage1Controllers(calendar=frame[["signal_date","trade_date"]],open_models=open_models,recovery_models=recovery_models)
pd.testing.assert_frame_equal(controller.audit(),objects["analysis/controller_audit"],check_dtype=False)
recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root,**prepared["action_plan"]["recovery_spec"])
action_spec=Stage1OpenActionValueSpec(root=root,**prepared["action_plan"]["action_spec"])
print('dated parent and minute source verified',len(receipt))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[28743:MainThread](2026-10-30 14:15:36,815) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[28743:MainThread](2026-10-30 14:15:36,817) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[28743:MainThread](2026-10-30 14:15:36,817) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[28743:MainThread](2026-10-30 14:15:36,893) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[28743:MainThread](2026-10-30 14:15:36,894) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[28743:MainThread](2026-10-30 14:15:36,895) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


dated parent and minute source verified 13


In [2]:
from src.backtest.stage1_capfree_lock_episode import replay_capfree_lock_episode
by_date={};quotes_by_date={};read_count=0
for item in plan["cases"]:
 day=pd.Timestamp(item["trade_date"])
 if day in by_date:continue
 checkpoint=capture_after_open(frame,event_day=day,controller_by_date=controller,
     recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared["selection"],plan=parent_plan)
 by_date[day]=checkpoint
 last=frame.iloc[checkpoint.index+4]
 rawbars=D.features(["SOXL","SOXX"],["$open","$close","$raw_open","$price_valid"],
     start_time=str(day.date())+" 09:31:00",end_time=str(pd.Timestamp(last.trade_date).date())+" 16:00:00",freq="1min")
 parts=[]
 for symbol in ("SOXL","SOXX"):
  x=rawbars.xs(symbol,level="instrument").copy();x.columns=[symbol+"_"+v.removeprefix("$") for v in x.columns];parts.append(x)
 quotes_by_date[day]=pd.concat(parts,axis=1).sort_index()
 read_count+=1
assert read_count==plan["budget"]["qlib_minute_reads"]
print('Qlib source reads',read_count)

Qlib source reads 3


In [3]:
max_error={k:0. for k in ('terminal_nav','max_drawdown','cash','quantity','fee')}
checks=[];nofill_checked=False
for item in plan['cases']:
 day=pd.Timestamp(item['trade_date']);checkpoint=by_date[day];quotes=quotes_by_date[day]
 bar_end=pd.Timestamp(str(day.date())+' '+item['bar_end_at']+':00');fill=bar_end+pd.Timedelta(minutes=2)
 late=bar_end>day+pd.Timedelta(hours=10,minutes=32)
 pos=book['native_positions'][day+pd.Timedelta(hours=10,minutes=32)]
 recovery_orders=book['orders'].loc[(book['orders'].trade_date==day)&(book['orders'].slot=='recovery')].to_dict('records')
 recovery_state=None
 q=checkpoint.state.q.copy();cash=float(checkpoint.state.cash);peak=float(checkpoint.state.peak);fees=deepcopy(checkpoint.state.fees)
 start_at=day+pd.Timedelta(hours=9,minutes=48)
 if late:
  recovery_state={'quantities':np.array([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ('SOXL','SOXX')],float),'cash':float(pos.get_cash()),'orders':recovery_orders}
  prior=quotes.loc[(quotes.index.normalize()==day)&(quotes.index>=start_at)&(quotes.index<=day+pd.Timedelta(hours=10,minutes=32))]
  prior_nav=prior[['SOXL_close','SOXX_close']].to_numpy(float)@q+cash
  peak=max(peak,float(np.max(prior_nav)))
  q=recovery_state['quantities'].copy();cash=recovery_state['cash']
  for order in recovery_orders:fees.commit(int(order['raw_shares']),stamp=order['stamp'],cap_applied=bool(order['cap_applied']))
  start_at=day+pd.Timedelta(hours=10,minutes=33)
 plane=compile_lock_price_plane(frame.iloc[checkpoint.index:checkpoint.index+5],quotes)
 mark=quotes.loc[bar_end];price=np.array([mark.SOXL_close,mark.SOXX_close],float)
 nav=float(q@price+cash);w=np.r_[q*price,cash]/nav
 targets={'SOXX':w+np.array([0.,.10,-.10]),'SOXL':w+np.array([.10/3,0.,-.10/3])}
 if item['name']=='halfday_late_mixed':targets['MIXED']=w+np.array([-.04,.08,-.04])
 for symbol,target in targets.items():
  bar=quotes.loc[fill+pd.Timedelta(minutes=1)]
  event={'trade_date':day,'bar_end_at':bar_end,'decision_at':bar_end+pd.Timedelta(minutes=1),'fill_at':fill,'desired_weights':target,'adjusted_fill_prices':np.array([bar.SOXL_open,bar.SOXX_open],float),'raw_fill_prices':np.array([bar.SOXL_raw_open,bar.SOXX_raw_open],float)}
  ref=replay_capfree_lock_episode(frame,checkpoint,quotes,event=event,recovery_state=recovery_state,horizon_sessions=5)
  fast=value_lock_event(plane,bar_end_at=bar_end,fill_at=fill,quantities=q,cash=cash,starting_peak=peak,fees=fees,desired_weights=target,start_at=start_at)
  assert fast['decision_nav']==ref['decision_nav'] and fast['label_available_at']==pd.Timestamp(frame.iloc[checkpoint.index+4].exit_date)+pd.Timedelta(hours=9,minutes=49)
  for arm in ('HOLD','EVENT'):
   x,y=fast[arm],ref[arm]
   local={'terminal_nav':abs(x['terminal_nav']-float(y['path'].iloc[-1].next_open_nav)),'max_drawdown':abs(x['max_drawdown']-y['terminal_available_mark_max_drawdown']),'cash':abs(x['cash']-y['ending_cash']),'quantity':float(np.max(np.abs(x['quantities']-y['ending_quantities']))),'fee':abs(x['fee_usd']-sum(o['total'] for o in y['orders']))}
   for key,value in local.items():max_error[key]=max(max_error[key],value)
   assert x['monthly_fee_shares']==y['monthly_fee_shares']
   assert [(o['symbol'],o['sell'],o['raw_shares']) for o in x['orders']]==[(o['symbol'],o['sell'],o['raw_shares']) for o in y['orders']]
  checks.append({'case':item['name'],'arm':symbol,'orders':len(fast['EVENT']['orders']),'recovery_orders_before_event':len(recovery_orders) if late else 0,'source_close':str(plane.quotes.loc[plane.quotes.index.normalize()==day].index.max()),'fast_terminal_nav_minus_hold':float(fast['EVENT']['terminal_nav']-fast['HOLD']['terminal_nav'])})
  if item['name']=='ordinary_early' and symbol=='SOXX':
   synthetic=dict(event,desired_weights=np.r_[q*event['adjusted_fill_prices'],cash]/float(q@event['adjusted_fill_prices']+cash))
   no_ref=replay_capfree_lock_episode(frame,checkpoint,quotes,event=synthetic,horizon_sessions=5)
   no_fast=value_lock_event(plane,bar_end_at=bar_end,fill_at=fill,quantities=q,cash=cash,starting_peak=peak,fees=fees,desired_weights=synthetic['desired_weights'],start_at=start_at)
   assert no_ref['event_status']=='no_whole_share_fill' and not no_fast['EVENT']['orders']
   assert abs(no_fast['EVENT']['terminal_nav']-no_fast['HOLD']['terminal_nav'])<1e-7
   assert abs(no_fast['EVENT']['max_drawdown']-no_fast['HOLD']['max_drawdown'])<1e-12
   nofill_checked=True
assert nofill_checked and len(checks)==9
assert max_error['terminal_nav']<=plan['maximum_terminal_nav_error_usd'] and max_error['max_drawdown']<=plan['maximum_drawdown_error']
assert max_error['cash']==max_error['quantity']==max_error['fee']==0
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'reference_module_sha256':plan['reference_module_sha256'],'vector_module_sha256':sha(root/plan['vector_module']),'fixed_cases':len(plan['cases']),'arms_checked':len(checks),'max_parity_errors':max_error,'synthetic_no_fill_identity_passed':nofill_checked,'case_arm_order_counts':[{k:v for k,v in x.items() if k!='fast_terminal_nav_minus_hold'} for x in checks],'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'vectorized_recovery_halfday_two_arm_parity_pass_not_bulk_labels'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(audit,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_lock_vectorized_clock_matrix_20261030",
  "plan_sha256": "d7d59cc3e4bb25f4b4ba69e1337ccba30dc4625e44c060994dad01af31b415a2",
  "reference_module_sha256": "8654a7c8adff7d5eff67013e24b0b299eb24c2d5cd5175c8676ba4787672819d",
  "vector_module_sha256": "aa1be2b7a9ece7e094360cf06e509d9cf28bf8ee456d33d44f01996fc48363b9",
  "fixed_cases": 4,
  "arms_checked": 9,
  "max_parity_errors": {
    "terminal_nav": 0.0,
    "max_drawdown": 0.0,
    "cash": 0.0,
    "quantity": 0.0,
    "fee": 0.0
  },
  "synthetic_no_fill_identity_passed": true,
  "case_arm_order_counts": [
    {
      "case": "ordinary_early",
      "arm": "SOXX",
      "orders": 1,
      "recovery_orders_before_event": 0,
      "source_close": "2019-12-19 16:00:00"
    },
    {
      "case": "ordinary_early",
      "arm": "SOXL",
      "orders": 1,
      "recovery_orders_before_event": 0,
      "source_close": "2019-12-19 16:00:00"
    },
    {
      "case": "halfday_late_mixed",
      "arm": "SOXX",
    

## 中文结论

**SOXX：无新候选账户或0/2/5bp结果。** 向量化核心在四个固定Qlib时钟的9个动作臂上对修正逐事件参考完全对账：HOLD/EVENT终端NAV、终端可用mark最大回撤、实际股数/现金、Tiered费用和月度股数计数最大差全0。覆盖SOXL/SOXX等名义beta买入、半日市SOXL卖出＋SOXX买入、2020恢复前/后父本状态及合成零整股单。恢复后向量化仅从10:33已完成bar起用已结算q/cash，起始高水位保留10:32前q0分钟路径；恢复前锁定两支仍共同取消稍后旧恢复。半日市用13:00实际闭市。

这只是9个机械身份点，不是全部332277个配对事件的经济标签。当前第一天09:48之前若有单根无效bar，价格平面只要求事件之后用到的bar有效；未来四个完整会话仍必须全部有效，和前卡来源门一致。10:30–10:32的恢复先后仍unknown。下一步冻结一次全分钟双臂路径标签的运行、Recorder及NAS预算，在完整意向上保留no-fill/unknown，2021/2022价值只入封存对象不用于特征选择。